In [1]:
import cvxpy as cp
import numpy as np
from scipy.optimize import minimize, Bounds
import time
from functools import partial

# 定义初始值
FLOPS = 624 * 0.55 * 1000000000000 # 混合精度（Tensor Core）FP16, 假设GPU利用率为0.35
s_r = 100
h = 5120
L = 40  # layers
V = 32000
epsilon = 10
SLO = 0.01
M_W = 26 
M_G = 30
R = 100

# 生成随机 s_r 数组, 长度为R，每个值的大小为100~1000
s_r = np.random.randint(100, 1000, R)
print("s_r:", s_r)

s_r: [261 546 449 304 429 342 865 354 377 802 247 267 308 225 388 547 560 407
 725 237 426 478 718 624 225 731 727 302 840 221 571 496 617 781 769 135
 820 841 432 576 225 812 437 205 874 795 210 128 185 150 820 857 183 615
 248 886 697 417 782 340 544 489 917 283 284 569 333 956 954 466 678 561
 593 405 900 155 389 852 797 561 433 771 316 997 288 113 485 480 571 844
 672 601 913 634 703 878 889 998 870 157]


In [14]:
st = time.time()
I = 2

# 定义T_r(l_i)
def T_r(l_i, r):
    A_r = 24 * s_r[r] * h**2 + 4 * s_r[r]**2 * h
    B_r = 24 * h**2 + 4 * h - 20 * s_r[r] * h**2 - 4 * s_r[r]**2 * h
    C_r = 2 * s_r[r] * h * V + 2 * h * V + 2 * epsilon
    return A_r * L + B_r * l_i + C_r

# 目标函数
def objective(vars, alpha=1e-4):
    b_i = vars[:I].astype(int)
    l = vars[I:]
    obj = 0
    for i in range(I):
        sum_T_r = sum(T_r(l[i], r) for r in range(b_i[i]))
        obj += sum_T_r / (b_i[i] * FLOPS)
    reg = alpha * np.sum(np.diff(l)**2)
    return obj + reg

# 约束条件：内存限制
# def constraint_memory(vars, i):
#     b_i = vars[:I].astype(int)
#     l = vars[I:]
#     current_b = int(b_i[i])
#     sum_sr = sum(s_r[r] for r in range(current_b)) if current_b > 0 else 0
#     if sum_sr == 0:
#         upper_bound = L
#     else:
#         memory_part = (M_G - M_W) / (2 * h * sum_sr * 2 / (1024**3))  # 确保单位转换正确
#         upper_bound = min(memory_part, L)
#     print(f"constraint_memory i={i}: b_i={current_b}, sum_sr={sum_sr}, upper_bound={upper_bound}, l={l[i]}")
#     return upper_bound - l[i]

def constraint_memory(vars, i):
    b_i = vars[:I].astype(int)
    l_i = vars[I:]
    return min((M_G - M_W) / (2 * h * sum(s_r[r] for r in range(b_i[i]))*2/(1024**3)), L) - l_i[i]

# 约束条件：总批次数
def constraint_total_batch(vars):
    return np.sum(vars[:I]) - R

# 约束条件：SLO
def constraint_total_SLO(vars, i):
    total_time = 0
    for k in range(i):
        b = int(vars[k])
        sum_T_r = sum(T_r(vars[I+k], r) for r in range(b))
        total_time += sum_T_r / FLOPS
    return SLO - total_time

# 初始化方法
def initialize_batch(I, R):
    base = R // I
    remainder = R % I
    initial_b = np.full(I, base)
    initial_b[:remainder] += 1
    return initial_b.astype(float)

# 初始猜测值
# initial_b = initialize_batch(I, R)
# initial_guess = np.concatenate((initial_b, np.full(I, 5)))  # 将 l_i 初始值设为 5

initial_guess = np.concatenate((np.full(I, R // I), np.random.uniform(2, L, I)))

# 边界条件
bounds = Bounds(
    np.concatenate((np.full(I, 1), np.full(I, 2))),
    np.concatenate((np.full(I, R - I + 1), np.full(I, L)))
)

# 约束条件
constraints = [
    {'type': 'eq', 'fun': constraint_total_batch}
]
for i in range(I):
    constraints.append({'type': 'ineq', 'fun': partial(constraint_memory, i=i)})
    constraints.append({'type': 'ineq', 'fun': partial(constraint_total_SLO, i=i)})

# 优化执行
result = minimize(
    objective, initial_guess, method='SLSQP',
    bounds=bounds, constraints=constraints,
    options={'maxiter': 500, 'disp': True}
)

# 后处理整数转换
final_b = np.round(result.x[:I]).astype(int)
remainder = R - np.sum(final_b)
if remainder != 0:
    indices = np.argsort(result.x[:I] - final_b)[-remainder:]
    final_b[indices] += 1

et = time.time()
print("Time elapsed:", (et - st)*1000, "ms")
print(f"Sum of batches: {np.sum(final_b)} (should be {R})")
print("Final batch sizes:", final_b)
print("Final l_i:", result.x[I:])
print("Final objective:", result.fun)

# # 检查约束条件
# print("Constraint checks:")
# for i in range(I):
#     print(f"Constraint memory {i}: {constraint_memory(result.x, i):.2f} >= 0")


Positive directional derivative for linesearch    (Exit mode 8)
            Current function value: 0.055514996611345145
            Iterations: 23
            Function evaluations: 118
            Gradient evaluations: 11
Time elapsed: 99.6243953704834 ms
Sum of batches: 100 (should be 100)
Final batch sizes: [50 50]
Final l_i: [14.6428559  17.96418609]
Final objective: 0.055514996611345145


In [18]:
st_0 = time.time()
opt_values = []
batch_sizes, cache_layers = [], []
for I in range(2, 6):
    print(f'----------------------------------I={I}----------------------------------')
    st = time.time()
    # 定义T_r(l_i)
    def T_r(l_i, r):
        A_r = 24 * s_r[r] * h**2 + 4 * s_r[r]**2 * h
        B_r = 24 * h**2 + 4 * h - 20 * s_r[r] * h**2 - 4 * s_r[r]**2 * h
        C_r = 2 * s_r[r] * h * V + 2 * h * V + 2 * epsilon
        return A_r * L + B_r * l_i + C_r

    # 定义目标函数
    def objective(vars):
        b_i = vars[:I].astype(int)  # 前I个变量是b_i
        l_i = vars[I:]  # 后I个变量是l_i
        obj = 0
        for i in range(I):
            sum_T_r = sum(T_r(l_i[i], r) for r in range(b_i[i]))
            obj += b_i[i] * FLOPS / sum_T_r
        return -obj  # 最大化问题转化为最小化问题

    # 定义约束条件
    def constraint1(vars, i):
        b_i = vars[:I].astype(int)
        l_i = vars[I:]
        sum_T_r = sum(T_r(l_i[i], r) for r in range(b_i[i]))
        return SLO - sum(1 / FLOPS * sum(T_r(l_i[k], r) for r in range(b_i[k])) for k in range(i + 1))

    def constraint2(vars, i):
        b_i = vars[:I].astype(int)
        l_i = vars[I:]
        return min((M_G - M_W) / (2 * h * sum(s_r[r] for r in range(b_i[i]))), L) - l_i[i]

    def constraint3(vars):
        b_i = vars[:I].astype(int)
        return np.sum(b_i) - R

    # 初始猜测值（加入随机性）
    initial_guess = np.concatenate((np.full(I, R // I), np.random.uniform(2, L, I)))  # b_i均匀分配，l_i随机初始化

    # 约束条件
    constraints = []
    for i in range(I):
        constraints.append({'type': 'ineq', 'fun': partial(constraint1, i=i)})
        constraints.append({'type': 'ineq', 'fun': partial(constraint2, i=i)})
    constraints.append({'type': 'eq', 'fun': constraint3})

    # 边界条件
    bounds = [(2, None)] * I + [(2, L)] * I  # b_i >= 1, l_i >= 1

    # 优化
    st = time.time()
    result = minimize(objective, initial_guess, method='SLSQP', bounds=bounds, constraints=constraints, options={'maxiter': 100, 'disp': True})

    # 输出结果
    et = time.time()
    print("Optimal b_i:", result.x[:I].astype(int))
    print("Optimal l_i:", result.x[I:])
    print("Objective value:", '{:.10f}'.format(-result.fun))
    print("Constraints:", [c['fun'](result.x) for c in constraints])
    print("Time:", (et - st) * 1000, 'ms')
    opt_values.append(-result.fun)
    batch_sizes.append(result.x[:I].astype(int))
    cache_layers.append(result.x[I:].astype(int))

# find the best I with the maximum objective value
best_I = np.argmax(opt_values) + 2
print(f'Best I={best_I}, Objective value={opt_values[best_I-2]}, batch_sizes={batch_sizes[best_I-2]}, cache_layers={cache_layers[best_I-2]}')

# 对于最终选择的cache_layers,让每个值都可以是4的倍数
cache_layers = cache_layers[best_I-2]
cache_layers = np.ceil(cache_layers / 4) * 4
print(f'Final cache_layers={cache_layers}')
et_0 = time.time()
print("Total time:", (et_0 - st_0) * 1000, 'ms')

----------------------------------I=2----------------------------------
Singular matrix C in LSQ subproblem    (Exit mode 6)
            Current function value: -157.7937971144981
            Iterations: 1
            Function evaluations: 5
            Gradient evaluations: 1
Optimal b_i: [50 50]
Optimal l_i: [39.03792034 12.51381152]
Objective value: 157.7937971145
Constraints: [-0.39030084822693356, -39.0379203280028, -1.9106241564601618, -12.513811501923522, 0]
Time: 11.763572692871094 ms
----------------------------------I=3----------------------------------
Singular matrix C in LSQ subproblem    (Exit mode 6)
            Current function value: -164.4499512619509
            Iterations: 1
            Function evaluations: 7
            Gradient evaluations: 1
Optimal b_i: [33 33 33]
Optimal l_i: [18.46524536 21.20090594 35.92299047]
Objective value: 164.4499512620
Constraints: [-0.8906611193830967, -18.465245340607094, -1.7092868236708552, -21.20090591569699, -2.0864351287305425,

In [19]:
# 定义T_r(l_i)
def T_r(l_i, r):
    A_r = 24 * s_r[r] * h**2 + 4 * s_r[r]**2 * h
    B_r = 24 * h**2 + 4 * h - 20 * s_r[r] * h**2 - 4 * s_r[r]**2 * h
    C_r = 2 * s_r[r] * h * V + 2 * h * V + 2 * epsilon
    return A_r * L + B_r * l_i + C_r

# 定义优化变量
l_i = cp.Variable(I, nonneg=True)  # l_i是非负实数变量

# 定义b_i为常数
b = R // I  # 因为所有b_i相同，所以b_i = R / I

# 定义目标函数
# 使用cvxpy.inv_pos来处理分数形式
# objective = cp.Maximize(cp.sum([b * FLOPS * cp.inv_pos(cp.sum([T_r(l_i[i], r) for r in range(b)])) for i in range(I)]))
# 将目标函数改成对数形式
objective = cp.Maximize(cp.sum([cp.log(b * FLOPS) - cp.log(cp.sum([T_r(l_i[i], r) for r in range(b)])) for i in range(I)]))

# 定义约束条件
constraints = []

# 约束1
for i in range(I):
    sum_T_r = cp.sum([T_r(l_i[i], r) for r in range(b)])
    constraints.append(1 / FLOPS * sum_T_r <= SLO - cp.sum([1 / FLOPS * cp.sum([T_r(l_i[k], r) for r in range(b)]) for k in range(i)]))

# 约束2
for i in range(I):
    constraints.append(l_i[i] <= cp.minimum((M_G - M_W) / (2 * h * cp.sum([s_r[r] for r in range(b)])), L))

# 优化问题
problem = cp.Problem(objective, constraints)

# 求解
problem.solve(solver=cp.SCS)

# 输出结果
print("Optimal l_i:", l_i.value)
print("Maximum Objective Value:", problem.value)
print("b_i (constant):", b)

DCPError: Problem does not follow DCP rules. Specifically:
The objective is not DCP, even though each sub-expression is.
You are trying to maximize a function that is convex.